In [16]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import torch
from torch import nn

from src.logreg import fitScaling, applyScaling
from src.evaluate import resultsRow,score

In [2]:
# cell 2
df = pd.read_parquet("../data/features/shots_features.parquet")
tbl = np.load("../data/features/player_tables.npz")
players = tbl["players"]
masks = tbl["masks"]

# nb 11 saved the tables in the same order as df
assert (tbl["ids"] == df["id"].to_numpy(dtype=str)).all()

# same 42 inputs as the goal models
drop = ["id", "match_id", "test", "fold", "goal", "x", "y"]
cols = []
for c in df.columns:
    if c not in drop:
        cols.append(c)
print(len(cols), "inputs")

42 inputs


In [3]:
# cell 3
tr = df[df["test"] == False]
te = df[df["test"] == True]
print(len(tr), "train shots,", len(te), "test shots")

74944 train shots, 24802 test shots


In [4]:
# cell 4
def makeX(a, b, deep=False):
    """X for the rows we train on (a) and some other rows (b). medians + scaling come from a only"""
    med = a[cols].median()
    Xa = a[cols].fillna(med).values
    Xb = b[cols].fillna(med).values

    mu, sd = fitScaling(Xa)
    Xa = applyScaling(Xa, mu, sd)
    Xb = applyScaling(Xb, mu, sd)

    if deep:
        # player tables are already 0-1, no scaling, just stuck on the end
        # df index is 0..n so it lines up with the tables
        Xa = np.hstack([Xa, players[a.index].reshape(len(a), -1), masks[a.index]])
        Xb = np.hstack([Xb, players[b.index].reshape(len(b), -1), masks[b.index]])

    return Xa, Xb

In [5]:
# cell 5
class Net(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.layers(x).squeeze(1)

In [6]:
# cell 6
def fitNet(model, X, y, epochs=10, bs=512, lr=0.001):
    """adam on log loss, shuffled batches. returns the loss of the last epoch"""
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossFn = nn.BCEWithLogitsLoss()  # does the sigmoid itself, more stable

    for ep in range(epochs):
        idx = torch.randperm(len(X))
        tot = 0
        for i in range(0, len(X), bs):
            b = idx[i:i + bs]
            opt.zero_grad()
            loss = lossFn(model(X[b]), y[b])
            loss.backward()
            opt.step()
            tot += loss.item() * len(b)

    return tot / len(X)


def predNet(model, X):
    """goal chance per row"""
    model.eval()
    with torch.no_grad():
        p = torch.sigmoid(model(torch.tensor(X, dtype=torch.float32)))
    model.train()
    return p.numpy()

In [7]:
# cell 7
def pickModel(deep):
    if deep:
        return DeepSet(len(cols))
    return Net(len(cols))


def cv(deep=False):
    """out-of-fold preds for train, then one model on all of train -> preds for train and test"""
    oof = np.zeros(len(tr))

    for k in range(1, 6):
        isK = (tr["fold"] == k).values
        Xa, Xb = makeX(tr[~isK], tr[isK], deep)
        model = pickModel(deep)
        loss = fitNet(model, Xa, tr["goal"].values[~isK])
        oof[isK] = predNet(model, Xb)
        print("fold", k, "loss", round(loss, 4))

    Xtr, Xte = makeX(tr, te, deep)
    model = pickModel(deep)
    fitNet(model, Xtr, tr["goal"].values)
    return oof, predNet(model, Xtr), predNet(model, Xte)

In [8]:
# cell 8
torch.manual_seed(2026)  # same starting weights every run

oofMlp, trMlp, teMlp = cv()
res = [resultsRow("MLP", tr["goal"], trMlp, tr["goal"], oofMlp, te["goal"], teMlp)]
pd.DataFrame(res).round(3)

fold 1 loss 0.2601
fold 2 loss 0.2627
fold 3 loss 0.262
fold 4 loss 0.2601
fold 5 loss 0.2601


,model,trainAuc,trainBrier,trainLogloss,validationAuc,validationBrier,validationLogloss,testAuc,testBrier,testLogloss
0,MLP,0.815,0.074,0.26,0.805,0.076,0.265,0.798,0.077,0.27


In [9]:
# cell 9
class DeepSet(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.n = n
        # phi: same small net for every player
        self.phi = nn.Sequential(
            nn.Linear(6, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
        )
        # rho: pooled players + shot features -> goal score
        self.rho = nn.Sequential(
            nn.Linear(64 + 64 + n, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        shot = x[:, :self.n]
        pl = x[:, self.n:self.n + 132].reshape(-1, 22, 6)  # 22 players x 6
        mk = x[:, self.n + 132:]

        h = self.phi(pl) * mk.unsqueeze(2)  # padding rows -> 0
        # sum + max over players so the order doesn't matter
        # relu is never negative so a 0 padding row can't win the max
        pooled = torch.cat([h.sum(1), h.max(1).values], dim=1)
        return self.rho(torch.cat([pooled, shot], dim=1)).squeeze(1)

In [10]:
# cell 10
torch.manual_seed(2026)

oofDs, trDs, teDs = cv(deep=True)
res.append(resultsRow("Deep Sets", tr["goal"], trDs, tr["goal"], oofDs, te["goal"], teDs))
pd.DataFrame(res).round(3)

fold 1 loss 0.2628
fold 2 loss 0.2653
fold 3 loss 0.2641
fold 4 loss 0.2628
fold 5 loss 0.2615


,model,trainAuc,trainBrier,trainLogloss,validationAuc,validationBrier,validationLogloss,testAuc,testBrier,testLogloss
0,MLP,0.815,0.074,0.260,0.805,0.076,0.265,0.798,0.077,0.27
1,Deep Sets,0.812,0.075,0.262,0.806,0.076,0.265,0.798,0.077,0.27


In [13]:
# cell 11
# deep sets shouldnt care about player order. shuffle the 22 rows, answer should stay the same
# true for any weights so an untrained model is fine for this
model = DeepSet(len(cols))
Xtr, Xte = makeX(tr, te, deep=True)
x = Xte[:100].copy()
before = predNet(model, x)

n = len(cols)
order = np.random.permutation(22)
pl = x[:, n:n + 132].reshape(-1, 22, 6)
x[:, n:n + 132] = pl[:, order].reshape(len(x), -1)
x[:, n + 132:] = x[:, n + 132:][:, order]
print("biggest change after shuffle:", np.abs(before - predNet(model, x)).max())

biggest change after shuffle: 5.9604645e-08


In [14]:
# ablation: drop the 9 hand-made freeze frame features
# deep sets still sees the raw players, the mlp doesn't. if deep sets holds up and the mlp drops,
# deep sets is picking up the frame on its own
frameCols = ["visible", "defendersInCone", "teammatesInCone", "nearestDefender", "defendersWithin3",
             "keeperMissing", "keeperInCone", "keeperToGoal", "keeperToShooter"]
allCols = cols
cols = []
for c in allCols:
    if c not in frameCols:
        cols.append(c)
print(len(cols), "inputs without the frame features")

torch.manual_seed(2026)
oof, trp, tep = cv()
res.append(resultsRow("MLP, no frame features", tr["goal"], trp, tr["goal"], oof, te["goal"], tep))

torch.manual_seed(2026)
oof, trp, tep = cv(deep=True)
res.append(resultsRow("Deep Sets, no frame features", tr["goal"], trp, tr["goal"], oof, te["goal"], tep))

cols = allCols  # full list back for anything after this
pd.DataFrame(res).round(3)[["model", "validationAuc", "testAuc"]]

33 inputs without the frame features
fold 1 loss 0.2685
fold 2 loss 0.272
fold 3 loss 0.272
fold 4 loss 0.2695
fold 5 loss 0.2701
fold 1 loss 0.2644
fold 2 loss 0.2667
fold 3 loss 0.267
fold 4 loss 0.2648
fold 5 loss 0.2649


,model,validationAuc,testAuc
0,MLP,0.805,0.798
1,Deep Sets,0.806,0.798
2,"MLP, no frame features",0.788,0.781
3,"Deep Sets, no frame features",0.802,0.794


In [12]:
# cell 12
pd.DataFrame(res).to_csv("../outputs/neural_results.csv", index=False)

# same layout as xg_oof: oof for train shots, final model for test shots
out = df[["id", "match_id", "test", "fold", "goal"]].copy()
out["mlp"] = 0.0
out["deepsets"] = 0.0
out.loc[tr.index, "mlp"] = oofMlp
out.loc[te.index, "mlp"] = teMlp
out.loc[tr.index, "deepsets"] = oofDs
out.loc[te.index, "deepsets"] = teDs
out.to_parquet("../outputs/neural_oof.parquet", index=False)
print(len(out), "rows saved")

99746 rows saved


In [17]:
# blend: plain average of tuned xgboost and deep sets, no weights picked so nothing is tuned on test
xg = pd.read_parquet("../outputs/xg_oof.parquet")[["id", "xgboost"]]
both = out.merge(xg, on="id")
both["blend"] = (both["xgboost"] + both["deepsets"]) / 2

va = both[both["test"] == False]
teb = both[both["test"] == True]
print("validation", score(va["goal"], va["blend"]))
print("test", score(teb["goal"], teb["blend"]))

validation {'auc': 0.8107626201044464, 'brier': 0.07497384962446575, 'logloss': 0.26221193764868267}
test {'auc': 0.8017650212474012, 'brier': 0.07649948494653129, 'logloss': 0.2675818351652092}
